In [ ]:
# 构造已报道异象的单变量与市值中性组合。
def get_quantile_labels(values, n=10):
    # 十分位标签：同月内按信号 rank 后 qcut；有效值不足十组时整列留 NaN。
    labels = pd.Series(
        np.nan,
        index=values.index,
        dtype=float,
    )
    valid = values.notna() & np.isfinite(values)
    if valid.sum() < n:
        return labels
    ranks = values.loc[valid].rank(method='first')
    labels.loc[valid] = (
        pd.qcut(ranks, q=n, labels=False).to_numpy() + 1
    )
    return labels

anomaly_specs = {
    'size': (monthly_market_value, 1, 10),
    'ep': (monthly_ep, 10, 1),
    'bm': (monthly_bm, 10, 1),
    'cp': (monthly_cp, 10, 1),
    'roe': (monthly_roe, 10, 1),
    'volatility': (monthly_volatility, 1, 10),
    'max': (monthly_max, 1, 10),
    'turnover': (monthly_turnover_12m, 1, 10),
    'abnormal_turnover': (monthly_abn_turnover_1m, 1, 10),
    'asset_growth': (monthly_asset_growth, 1, 10),
    'accruals': (monthly_accrual, 1, 10),
    'noa': (monthly_noa, 1, 10),
    'illiquidity': (monthly_illiquidity, 10, 1),
}

def get_anomaly_returns(monthly_final, size_neutral=False):
    # 异象多空收益：单变量分支 = 全样本内十分位（对标 get_groups_uncon）；
    # 市值中性分支 = 先市值十分位、再每个市值组内信号十分位，池化后按信号组
    # 重新市值加权（与 portfolio_5 的 5×5 格内平均刻意不同）。
    return_list = []
    portfolio_list = []
    for name, (signal, long_leg, short_leg) in anomaly_specs.items():
        if size_neutral and name == 'size':
            continue
        data = monthly_final.dropna(
            subset=[signal, monthly_market_value]
        ).copy()
        if name in ['ep','cp']:
            data = data[data[signal].gt(0)]
        has_finite_signal = np.isfinite(data[signal])
        has_positive_market_value = data[monthly_market_value].gt(0)
        data = data.loc[has_finite_signal & has_positive_market_value]
        print('异象 {} / 中性 {} / 可排序行 {}'.format(name,size_neutral,len(data)))
        if data.empty:
            return_list.append(pd.Series(dtype=float,name=name))
            continue
        if size_neutral:
            data['size_group'] = data.groupby(ym)[monthly_market_value].transform(
                lambda values: get_quantile_labels(values, 10)
            )
            data['anomaly_group'] = data.groupby(
                [ym, 'size_group']
            )[signal].transform(
                lambda values: get_quantile_labels(values, 10)
            )
        else:
            data['anomaly_group'] = data.groupby(ym)[signal].transform(
                lambda values: get_quantile_labels(values, 10)
            )
        ranked = data.dropna(subset=['anomaly_group'])
        if ranked.empty:
            return_list.append(pd.Series(dtype=float,name=name))
            print('{} 没有足够观测形成十组，保留缺数状态'.format(name))
            continue
        portfolio_columns = [monthly_market_value, monthly_excess_ret]
        ports = ranked.groupby([ym, 'anomaly_group'])[portfolio_columns]
        ports = ports.apply(get_vw_return).unstack()
        ports = ports.reindex(columns=range(1, 11))
        return_list.append((ports[long_leg]-ports[short_leg]).rename(name))
        ports.columns = ['{}_{}'.format(name,i) for i in ports.columns]
        portfolio_list.append(ports)
    return pd.concat(return_list, axis=1), pd.concat(portfolio_list, axis=1)


In [ ]:
anomaly_uncon, portfolios_uncon = get_anomaly_returns(
    monthly_final,
    size_neutral=False,
)

anomaly_neutral, portfolios_neutral = get_anomaly_returns(
    monthly_final,
    size_neutral=True,
)


# 最后按五个交叠持有队列构造日频反转，并汇总到月度。
# src_jup 没有日频反转实现（features_3 备注：反转用月收益排序即可构造）；
# 五份队列、信号先 shift 一个市场交易日、缺月重启的规则见 learning/Size_and_Value_in_China/02。

